In [65]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, TransformerMixin
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk
import joblib
import re

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_and_lemmatize(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)  # remove newlines
    text = re.sub(r'https?://\S+', 'url', text)  # replace URLs with 'url'
    text = re.sub(r'\d{5,}', 'number', text)  # replace long numbers (like phone numbers) with 'number'
    text = re.sub(r'[^a-z0-9£$€\s]', '', text)  #  Keep letters, numbers, currency symbols, spaces
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

In [ ]:
# Custom transformer for text cleaning
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return [clean_and_lemmatize(text) for text in X]

# Main Trainer Class
class BinaryClassifierTrainer:
    def __init__(self, connection_string, model_path="./context_binary_classifier.pkl"):
        self.connection_string = connection_string
        self.model_path = model_path

        # Class mapping for reference (optional)
        self.class_mapping = {0: "Email", 1: "SMS"}

        # Define pipeline
        self.pipeline = Pipeline([
            ('preprocess', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),  # unigram + bigram
            ('smote', SMOTE(random_state=50)),
            ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
        ])
        # ('clf', MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42))
        # ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))

    def train(self, df):
        X = df['content']  # Assuming DB column name is 'text'
        y = df['class']  # Already numeric: 0, 1, 2

        # Split dataset
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=50, stratify=y
        )

        # Train pipeline
        self.pipeline.fit(X_train, y_train)

        # Evaluate with class names using mapping
        y_pred = self.pipeline.predict(X_test)
        y_test_labels = [self.class_mapping[label] for label in y_test]
        y_pred_labels = [self.class_mapping[label] for label in y_pred]
        acc = accuracy_score(y_test, y_pred)
        print("Accuracy Score:", round(acc, 4))

        print(classification_report(y_test_labels, y_pred_labels, target_names=list(self.class_mapping.values())))
    
    def save_model(self):
        joblib.dump({
            "pipeline": self.pipeline,
            "class_mapping": self.class_mapping
        }, self.model_path)
        print(f"Model and class mapping saved to {self.model_path}")

In [ ]:
# Load data from database
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Database connection details
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Dataset folder path
DATASET_PATH = os.getenv('DATASET_PATH')

# Create the connection engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
# Load the whole table
email_df = pd.read_sql("SELECT * FROM email_detection_dataset", engine)
email_df = email_df[["content", "class"]]
email_df["class"] = 0 # 0 for email

sms_df = pd.read_sql("SELECT * FROM accumulated_training_data", engine)
sms_df = sms_df[["text", "label"]]
sms_df = sms_df.rename(columns={'text': 'content', 'label': 'class'})
sms_df["class"] = 1 # 1 for SMS

df = pd.concat([email_df, sms_df], ignore_index=True)


In [ ]:
# Build the connection string from env variables
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

trainer = BinaryClassifierTrainer(connection_string)
trainer.train(df)
trainer.save_model()

c:\Users\malon\anaconda3\envs\tensorflow\lib\site-packages\sklearn\linear_model\_logistic.py:1262: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy Score: 0.9335
              precision    recall  f1-score   support

       Email       0.96      0.90      0.93      2386
         SMS       0.91      0.96      0.94      2548

    accuracy                           0.93      4934
   macro avg       0.94      0.93      0.93      4934
weighted avg       0.93      0.93      0.93      4934

Model and class mapping saved to ./SMS_Email_classifier.pkl
